# Content-Based Recommendation

Build movie features from genres and release decade, then check whether they help rank rare movies. Feature choices are selected on validation only.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))   # notebooks/ -> project root -> scripts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recutils import load_split, to_sets

NAME, K, REL, SEED = "sample_10pct", 10, 4.0, 42

S = load_split(NAME)
train, val, test = S["train"], S["val"], S["test"]
n_users = int(S["user_map"].u.max()) + 1
n_items = int(S["item_map"].i.max()) + 1
print(f"{n_users:,} users | {n_items:,} movies in the modelled catalogue")

movies = pd.read_parquet(Path("../data/interim") / NAME / "movies.parquet")
meta   = S["item_map"].merge(movies, on="movieId", how="left").set_index("i").sort_index()
assert (meta.index.values == np.arange(n_items)).all() and meta.title.notna().all()


20,093 users | 17,241 movies in the modelled catalogue


## Rare-movie coverage

Measure how much of the validation set comes from movies with little or no positive training history.


In [2]:
n_pos = np.bincount(train.i[train.rating >= REL], minlength=n_items)

bins   = [-1, 0, 4, 19, 99, 499, np.inf]
labels = ["0", "1-4", "5-19", "20-99", "100-499", "500+"]
bucket = pd.Series(pd.cut(n_pos, bins=bins, labels=labels), name="bucket")

def share_table(rel_df):
    b = pd.Series(bucket.values[rel_df.i.values])
    n_int = b.value_counts().reindex(labels)
    n_usr = pd.DataFrame({"u": rel_df.u.values, "b": b.values}).groupby("b", observed=False)["u"].nunique().reindex(labels)
    return n_int, n_usr

val_rel, test_rel = val[val.rating >= REL], test[test.rating >= REL]
v_int, v_usr = share_table(val_rel)
t_int, t_usr = share_table(test_rel)

summary = pd.DataFrame({
    "movies": bucket.value_counts().reindex(labels),
    "% of catalogue": (100 * bucket.value_counts().reindex(labels) / n_items).round(1),
    "relevant val interactions": v_int,
    "% of relevant val": (100 * v_int / v_int.sum()).round(2),
    "users (val)": v_usr,
    "relevant test interactions": t_int,
    "% of relevant test": (100 * t_int / t_int.sum()).round(2),
    "users (test)": t_usr,
})
summary

,movies,% of catalogue,relevant val interactions,% of relevant val,users (val),relevant test interactions,% of relevant test,users (test)
0,1095,6.4,80,0.11,77,121,0.16,116
1-4,5666,32.9,784,1.04,625,985,1.28,796
5-19,4836,28.0,2355,3.13,1761,2924,3.80,2220
20-99,3299,19.1,8460,11.23,5345,9646,12.54,6041
100-499,1685,9.8,21356,28.34,10663,22204,28.86,11236
500+,660,3.8,42318,56.16,14988,41049,53.36,15018


## Metadata coverage


In [3]:
meta["year"]      = meta.title.str.extract(r"\((\d{4})\)\s*$")[0].astype("float")
meta["has_genre"] = meta.genres.ne("(no genres listed)")
meta["has_year"]  = meta.year.notna()
meta["n_genres"]  = meta.genres.where(meta.has_genre, "").str.count(r"\|").add(1).where(meta.has_genre, 0)
meta["bucket"]    = bucket.values

cov = meta.groupby("bucket", observed=False).agg(
    movies=("title", "size"),
    pct_with_genre=("has_genre", "mean"),
    pct_with_year=("has_year", "mean"),
    mean_n_genres=("n_genres", "mean"),
)
cov[["pct_with_genre", "pct_with_year"]] = (100 * cov[["pct_with_genre", "pct_with_year"]]).round(1)
cov["mean_n_genres"] = cov["mean_n_genres"].round(2)
cov.reindex(labels)

,movies,pct_with_genre,pct_with_year,mean_n_genres
bucket,,,,
0,1095,98.7,99.8,2.07
1-4,5666,98.9,99.7,1.98
5-19,4836,99.3,99.7,2.02
20-99,3299,99.8,99.7,2.22
100-499,1685,99.8,99.8,2.51
500+,660,100.0,100.0,2.92


The evaluation is concentrated in popular movies. Movies with fewer than 20 positive training ratings make up 67% of the catalogue but account for 4.3% of relevant validation interactions (5.2% on test). Movies with fewer than 5 positives make up 39% of the catalogue and account for 1.2% of validation interactions (1.4% on test).

Genres and release years are available for almost the entire catalogue, including rare movies. The limitation is that many movies share the same genre-and-decade combination, so metadata may not distinguish them well.

We use two slices: **tail** (<20 positive training ratings) as the main group, and **cold** (<5) as a harder secondary group. View A ranks against the full catalogue; View B ranks only within the slice. Test slice membership is recalculated from train + validation. Metrics are averaged across users with at least one relevant item in the slice, with paired user-level bootstrap intervals.

## Define the rarity groups


In [4]:
def positives(df):
    return np.bincount(df.i[df.rating >= REL], minlength=n_items)

n_pos_val  = positives(train)                         # what the model knows when we tune on validation
n_pos_test = positives(pd.concat([train, val]))       # what the final model knows when we test

SLICES = {"cold": 5, "tail": 20}                      # fewer than this many positives
in_slice = {("val", s): n_pos_val < t for s, t in SLICES.items()}
in_slice.update({("test", s): n_pos_test < t for s, t in SLICES.items()})

def slice_table(rel_df, split):
    rows = {}
    for s, t in SLICES.items():
        movies_in = in_slice[(split, s)]
        hit = movies_in[rel_df.i.values]
        rows[f"{s} (<{t} positives)"] = {
            "movies": int(movies_in.sum()),
            "relevant interactions": int(hit.sum()),
            "% of all relevant": round(100 * hit.mean(), 2),
            "users with >=1": int(np.unique(rel_df.u.values[hit]).size),
            "random Recall@10": round(10 / movies_in.sum(), 4),
        }
    return pd.DataFrame(rows).T.astype({"movies": int, "relevant interactions": int, "users with >=1": int})

print("validation (movie counts from train)")
display(slice_table(val_rel, "val"))
print("test (movie counts from train + validation)")
display(slice_table(test_rel, "test"))

validation (movie counts from train)


,movies,relevant interactions,% of all relevant,users with >=1,random Recall@10
cold (<5 positives),6761,864,1.15,676,0.0015
tail (<20 positives),11597,3219,4.27,2144,0.0009


test (movie counts from train + validation)


,movies,relevant interactions,% of all relevant,users with >=1,random Recall@10
cold (<5 positives),6572,1033,1.34,814,0.0015
tail (<20 positives),11456,3802,4.94,2587,0.0009


## Build genre and decade features


In [5]:
genres = meta.genres.str.get_dummies("|").drop(columns="(no genres listed)", errors="ignore").astype("int8")

decade  = (meta.year // 10 * 10).clip(lower=1910)           # anything before 1920 goes into one bucket
decades = pd.get_dummies(decade.map(lambda d: f"{int(d)}s" if pd.notna(d) else None)).astype("int8")

print("genre matrix:", genres.shape, "|", int(genres.sum(axis=1).eq(0).sum()), "movies with no genre")
print("decade matrix:", decades.shape, "|", int(decades.sum(axis=1).eq(0).sum()), "movies with no year")
genres.sum().sort_values(ascending=False)

genre matrix: (17241, 19) | 125 movies with no genre
decade matrix: (17241, 12) | 47 movies with no year


Drama          8082
Comedy         5819
Thriller       3284
Action         2676
Romance        2541
Crime          1959
Horror         1888
Adventure      1807
Sci-Fi         1495
Fantasy        1205
Children       1081
Animation      1068
Mystery        1050
Documentary     946
War             670
Musical         532
Western         325
IMAX            173
Film-Noir       164
dtype: int64

The binary features are weighted and normalized before comparing movies.


In [6]:
B = pd.concat([genres, decades], axis=1)
print("Binary genre + decade features:", B.shape)


Binary genre + decade features: (17241, 31)


### Identical feature vectors

Movies with the same genre and decade combination cannot be distinguished by this representation alone.


In [7]:
vec_id = pd.util.hash_pandas_object(B, index=False)         # one hash per row; same row -> same hash
same   = vec_id.map(vec_id.value_counts()) - 1              # how many OTHER movies share this vector

print("distinct vectors:", vec_id.nunique(), "of", n_items, "movies")
print("sizes of the 5 biggest groups of identical movies:", vec_id.value_counts().head(5).tolist())

meta["shared"] = same > 0
meta.groupby("bucket", observed=False).agg(
    movies=("title", "size"),
    pct_sharing_a_vector=("shared", lambda s: round(100 * s.mean(), 1)),
)

distinct vectors: 3052 of 17241 movies
sizes of the 5 biggest groups of identical movies: [587, 531, 464, 391, 329]


,movies,pct_sharing_a_vector
bucket,,
0,1095,92.1
1-4,5666,90.9
5-19,4836,91.5
20-99,3299,89.1
100-499,1685,86.8
500+,660,81.2


In [8]:
meta["identical_others"] = same        # movies with exactly the same vector as this one
meta.groupby("bucket", observed=False).agg(
    movies=("title", "size"),
    median_identical_others=("identical_others", "median"),
    pct_with_10_plus=("identical_others", lambda s: round(100 * (s >= 10).mean(), 1)),
)

,movies,median_identical_others,pct_with_10_plus
bucket,,,
0,1095,26.0,66.8
1-4,5666,29.0,69.3
5-19,4836,31.0,69.0
20-99,3299,24.0,64.8
100-499,1685,15.0,57.0
500+,660,8.0,45.2


## Validation setup


In [9]:
import scipy.sparse as sp
from recutils import rank_topk

def seen_matrix(*dfs):
    """Sparse 0/1 matrix of (user, movie) pairs a user already has; used to hide them from recommendations."""
    d = pd.concat(dfs)
    return sp.csr_matrix((np.ones(len(d), dtype=np.float32), (d.u.values, d.i.values)),
                         shape=(n_users, n_items))

seen_train = seen_matrix(train)
rel_val = to_sets(val_rel)          # user -> set of validation movies they rated >= 4
print("validation users with >=1 relevant movie:", len(rel_val))

validation users with >=1 relevant movie: 18392


## Shared ranking and evaluation helpers

The same user-level metrics and tie-breaking seeds are used for every feature configuration.


In [10]:
def per_user(ranked, relevant):
    discounts = 1.0 / np.log2(np.arange(2, K + 2))
    recalls, ndcgs = [], []
    for user_id, relevant_items in relevant.items():
        top = ranked[user_id][:K]
        hits = np.asarray([item in relevant_items for item in top], dtype=float)
        recalls.append(float(hits.sum() / len(relevant_items)))
        ideal = discounts[:min(len(relevant_items), K)].sum()
        ndcgs.append(float((hits * discounts[:len(hits)]).sum() / ideal))
    return {"Recall@10": np.asarray(recalls), "NDCG@10": np.asarray(ndcgs)}

def slice_rel(relevant, mask):
    sliced = {
        user_id: {item for item in items if mask[item]}
        for user_id, items in relevant.items()
    }
    return {user_id: items for user_id, items in sliced.items() if items}

counts = np.bincount(train.i, minlength=n_items).astype(np.float32)
rng = np.random.default_rng(SEED)

def pop_fn(user_batch):
    return np.tile(counts, (len(user_batch), 1)) + 1e-3 * rng.random(
        (len(user_batch), n_items), dtype=np.float32
    )

def random_fn(user_batch):
    return rng.random((len(user_batch), n_items), dtype=np.float32)

MODELS = {"random": random_fn, "popularity": pop_fn}


## Content feature ablation

Test genre-only, decade-only and combined representations, with IDF weighting on and off. Tail View B NDCG@10 is the primary selection metric; cold-slice results are reported as a secondary check.


In [11]:
def make_feature_matrix(
    genre_weight=1.0,
    decade_weight=1.0,
    use_idf=True,
):
    # B already contains genres followed by decade features.
    binary_features = B.to_numpy(dtype=np.float32)

    # Begin with equal weights for every feature.
    weights = np.ones(B.shape[1], dtype=np.float32)

    # The first columns represent genres.
    n_genre_features = genres.shape[1]
    weights[:n_genre_features] *= genre_weight

    # The remaining columns represent decades.
    weights[n_genre_features:] *= decade_weight

    # Optionally downweight common features.
    if use_idf:
        feature_counts = B.sum(axis=0).to_numpy(dtype=np.float32)
        idf_weights = np.log(
            n_items / np.maximum(feature_counts, 1.0)
        )
        weights *= idf_weights

    # Apply the selected weights to every movie.
    X_features = binary_features * weights

    # Normalize each movie vector to unit length.
    row_norms = np.linalg.norm(
        X_features, axis=1, keepdims=True
    )
    X_features = np.divide(
        X_features,
        row_norms,
        out=np.zeros_like(X_features),
        where=row_norms > 0,
    )

    return X_features

In [12]:
def build_profiles_for(df, X_features):
    # Keep positive ratings from the supplied dataset.
    liked = df[df.rating >= REL]

    # Construct the user-movie interaction matrix.
    L = sp.csr_matrix(
        (
            np.ones(len(liked), dtype=np.float32),
            (liked.u.values, liked.i.values),
        ),
        shape=(n_users, n_items),
    )

    # Count how many liked movies each user has.
    n_liked = np.asarray(L.sum(axis=1)).ravel()

    # Calculate 1 / number of liked movies.
    inv = np.divide(
        1.0,
        n_liked,
        out=np.zeros_like(n_liked),
        where=n_liked > 0,
    )

    # Average the feature vectors of each user's liked movies.
    profiles = sp.diags(inv) @ L @ X_features

    return np.asarray(profiles, dtype=np.float32), n_liked

In [13]:
def make_content_scorer(X_features, user_profiles):
    item_features = np.asarray(
        X_features, dtype=np.float32
    )
    user_profiles = np.asarray(
        user_profiles, dtype=np.float32
    )

    def score(user_batch):
        scores = (
            user_profiles[user_batch] @ item_features.T
        )

        # Break exact or near-identical-score ties reproducibly.
        return scores + 1e-6 * rng.random(
            scores.shape, dtype=np.float32
        )

    return score

In [14]:
FEATURE_VARIANTS = [
    ("genres_only", 1.0, 0.0),
    ("decade_only", 0.0, 1.0),
    ("both_decade_0.25", 1.0, 0.25),
    ("both_decade_0.5", 1.0, 0.5),
    ("both_decade_1.0", 1.0, 1.0),
]

CONFIGS = []

for variant, genre_weight, decade_weight in FEATURE_VARIANTS:
    for use_idf in (False, True):
        CONFIGS.append({
            "name": (
                f"{variant}_idf_"
                f"{'on' if use_idf else 'off'}"
            ),
            "genre_weight": genre_weight,
            "decade_weight": decade_weight,
            "use_idf": use_idf,
        })

config_table = pd.DataFrame(CONFIGS)

display(config_table)

assert len(CONFIGS) == 10

DEFAULT_CONFIG = "both_decade_1.0_idf_on"

assert sum(
    cfg["name"] == DEFAULT_CONFIG for cfg in CONFIGS
) == 1

,name,genre_weight,decade_weight,use_idf
0,genres_only_idf_off,1.0,0.00,False
1,genres_only_idf_on,1.0,0.00,True
2,decade_only_idf_off,0.0,1.00,False
3,decade_only_idf_on,0.0,1.00,True
4,both_decade_0.25_idf_off,1.0,0.25,False
5,both_decade_0.25_idf_on,1.0,0.25,True
6,both_decade_0.5_idf_off,1.0,0.50,False
7,both_decade_0.5_idf_on,1.0,0.50,True
8,both_decade_1.0_idf_off,1.0,1.00,False
9,both_decade_1.0_idf_on,1.0,1.00,True


In [15]:
N_TIE_SEEDS = 10

def eval_slice(slice_name, restrict, names):
    global rng                                   # the model functions read this generator
    mask = in_slice[("val", slice_name)]
    rel = slice_rel(rel_val, mask)
    out = {}
    for name in names:
        fn = MODELS[name]
        f = (lambda ub, fn=fn: np.where(mask, fn(ub), -np.inf)) if restrict else fn
        runs = []
        for seed in range(N_TIE_SEEDS):
            rng = np.random.default_rng(seed)    # fresh, fixed generator per run
            runs.append(per_user(rank_topk(f, rel.keys(), seen_train, K), rel))
        out[name] = {met: np.mean([r[met] for r in runs], axis=0) for met in runs[0]}
    return out, len(rel)

def ci(vals, idx):
    m = vals[idx].mean(axis=1)
    return vals.mean(), *np.percentile(m, [2.5, 97.5])


In [16]:
ablation_runs = {}

for number, cfg in enumerate(CONFIGS, start=1):
    X_cfg = make_feature_matrix(
        genre_weight=cfg["genre_weight"],
        decade_weight=cfg["decade_weight"],
        use_idf=cfg["use_idf"],
    )
    profiles_cfg, n_liked_cfg = build_profiles_for(train, X_cfg)
    scorer = make_content_scorer(X_cfg, profiles_cfg)

    MODELS["content"] = scorer
    tail_metrics, n_tail = eval_slice("tail", restrict=True, names=["content"])
    cold_metrics, n_cold = eval_slice("cold", restrict=True, names=["content"])

    ablation_runs[cfg["name"]] = {
        "tail": tail_metrics["content"],
        "cold": cold_metrics["content"],
        "n_tail_users": n_tail,
        "n_cold_users": n_cold,
    }
    print(
        f"{number}/{len(CONFIGS)} {cfg['name']}: "
        f"tail NDCG@10={tail_metrics['content']['NDCG@10'].mean():.5f}, "
        f"cold NDCG@10={cold_metrics['content']['NDCG@10'].mean():.5f}"
    )


1/10 genres_only_idf_off: tail NDCG@10=0.00263, cold NDCG@10=0.00452
2/10 genres_only_idf_on: tail NDCG@10=0.00335, cold NDCG@10=0.00501
3/10 decade_only_idf_off: tail NDCG@10=0.00101, cold NDCG@10=0.00164
4/10 decade_only_idf_on: tail NDCG@10=0.00101, cold NDCG@10=0.00164
5/10 both_decade_0.25_idf_off: tail NDCG@10=0.00475, cold NDCG@10=0.00425
6/10 both_decade_0.25_idf_on: tail NDCG@10=0.00531, cold NDCG@10=0.00609
7/10 both_decade_0.5_idf_off: tail NDCG@10=0.00549, cold NDCG@10=0.00471
8/10 both_decade_0.5_idf_on: tail NDCG@10=0.00541, cold NDCG@10=0.00449
9/10 both_decade_1.0_idf_off: tail NDCG@10=0.00584, cold NDCG@10=0.00565
10/10 both_decade_1.0_idf_on: tail NDCG@10=0.00529, cold NDCG@10=0.00563


In [17]:
# The default configuration is the reference for paired comparisons.
default_tail = ablation_runs[DEFAULT_CONFIG]["tail"]["NDCG@10"]
default_cold = ablation_runs[DEFAULT_CONFIG]["cold"]["NDCG@10"]

# Check that every configuration was evaluated on the same number of users.
tail_lengths = {len(ablation_runs[cfg["name"]]["tail"]["NDCG@10"]) for cfg in CONFIGS}
cold_lengths = {len(ablation_runs[cfg["name"]]["cold"]["NDCG@10"]) for cfg in CONFIGS}
assert len(tail_lengths) == 1 and len(cold_lengths) == 1

n_tail_users = len(default_tail)
n_cold_users = len(default_cold)
bootstrap_rng = np.random.default_rng(SEED)
N_BOOTSTRAP = 1000

# Each row contains the user indices for one bootstrap resample.
tail_idx = bootstrap_rng.integers(0, n_tail_users, size=(N_BOOTSTRAP, n_tail_users))
cold_idx = bootstrap_rng.integers(0, n_cold_users, size=(N_BOOTSTRAP, n_cold_users))

print(f"Tail slice: {n_tail_users:,} users | Cold slice: {n_cold_users:,} users")
print(f"Bootstrap resamples: {N_BOOTSTRAP:,}")

Tail slice: 2,144 users | Cold slice: 676 users
Bootstrap resamples: 1,000


In [18]:
rows = []

for cfg in CONFIGS:
    name = cfg["name"]
    tail = ablation_runs[name]["tail"]
    cold = ablation_runs[name]["cold"]

    tail_ndcg = tail["NDCG@10"]
    cold_ndcg = cold["NDCG@10"]

    tail_mean, tail_low, tail_high = ci(tail_ndcg, tail_idx)
    cold_mean, cold_low, cold_high = ci(cold_ndcg, cold_idx)

    # Subtract user by user, then bootstrap these paired differences.
    tail_delta = tail_ndcg - default_tail
    delta_mean, delta_low, delta_high = ci(tail_delta, tail_idx)

    rows.append({
        "config": name,
        "genre_weight": cfg["genre_weight"],
        "decade_weight": cfg["decade_weight"],
        "use_idf": cfg["use_idf"],
        "tail_NDCG@10": tail_mean,
        "tail_NDCG_CI_low": tail_low,
        "tail_NDCG_CI_high": tail_high,
        "tail_delta_vs_default": delta_mean,
        "delta_CI_low": delta_low,
        "delta_CI_high": delta_high,
        "tail_Recall@10": tail["Recall@10"].mean(),
        "cold_NDCG@10": cold_mean,
        "cold_NDCG_CI_low": cold_low,
        "cold_NDCG_CI_high": cold_high,
        "cold_Recall@10": cold["Recall@10"].mean(),
    })

results_df = (
    pd.DataFrame(rows)
    .sort_values("tail_NDCG@10", ascending=False)
    .reset_index(drop=True)
)

# A compact display is easier to compare in the notebook; the CSV keeps
# the numerical lower and upper bounds in separate columns.
display_df = pd.DataFrame({
    "Configuration": results_df["config"],
    "Tail NDCG@10": results_df["tail_NDCG@10"].map(lambda x: f"{x:.5f}"),
    "Tail NDCG 95% CI": results_df.apply(lambda r: f"[{r['tail_NDCG_CI_low']:.5f}, {r['tail_NDCG_CI_high']:.5f}]", axis=1),
    "Delta vs default": results_df["tail_delta_vs_default"].map(lambda x: f"{x:+.5f}"),
    "Paired delta 95% CI": results_df.apply(lambda r: f"[{r['delta_CI_low']:.5f}, {r['delta_CI_high']:.5f}]", axis=1),
    "Tail Recall@10": results_df["tail_Recall@10"].map(lambda x: f"{x:.5f}"),
    "Cold NDCG@10": results_df["cold_NDCG@10"].map(lambda x: f"{x:.5f}"),
    "Cold NDCG 95% CI": results_df.apply(lambda r: f"[{r['cold_NDCG_CI_low']:.5f}, {r['cold_NDCG_CI_high']:.5f}]", axis=1),
    "Cold Recall@10": results_df["cold_Recall@10"].map(lambda x: f"{x:.5f}"),
})
display(display_df)

,Configuration,Tail NDCG@10,Tail NDCG 95% CI,Delta vs default,Paired delta 95% CI,Tail Recall@10,Cold NDCG@10,Cold NDCG 95% CI,Cold Recall@10
0,both_decade_1.0_idf_off,0.00584,"[0.00390, 0.00795]",+0.00055,"[-0.00087, 0.00210]",0.01153,0.00565,"[0.00236, 0.00943]",0.01114
1,both_decade_0.5_idf_off,0.00549,"[0.00365, 0.00765]",+0.00021,"[-0.00143, 0.00195]",0.01043,0.00471,"[0.00185, 0.00774]",0.01053
2,both_decade_0.5_idf_on,0.00541,"[0.00364, 0.00746]",+0.00013,"[-0.00099, 0.00128]",0.01067,0.00449,"[0.00180, 0.00744]",0.01001
3,both_decade_0.25_idf_on,0.00531,"[0.00329, 0.00776]",+0.00002,"[-0.00182, 0.00210]",0.00918,0.00609,"[0.00264, 0.01043]",0.01240
4,both_decade_1.0_idf_on,0.00529,"[0.00353, 0.00726]",+0.00000,"[0.00000, 0.00000]",0.01046,0.00563,"[0.00252, 0.00902]",0.01349
5,both_decade_0.25_idf_off,0.00475,"[0.00302, 0.00680]",-0.00053,"[-0.00228, 0.00119]",0.00836,0.00425,"[0.00146, 0.00735]",0.00962
6,genres_only_idf_on,0.00335,"[0.00182, 0.00526]",-0.00194,"[-0.00387, 0.00017]",0.00581,0.00501,"[0.00179, 0.00896]",0.01044
7,genres_only_idf_off,0.00263,"[0.00132, 0.00433]",-0.00265,"[-0.00467, -0.00063]",0.00379,0.00452,"[0.00123, 0.00846]",0.00810
8,decade_only_idf_off,0.00101,"[0.00075, 0.00128]",-0.00428,"[-0.00626, -0.00250]",0.00212,0.00164,"[0.00099, 0.00240]",0.00328
9,decade_only_idf_on,0.00101,"[0.00075, 0.00128]",-0.00428,"[-0.00626, -0.00250]",0.00212,0.00164,"[0.00099, 0.00240]",0.00328


In [19]:
best = results_df.iloc[0]
default_row = results_df.loc[results_df["config"] == DEFAULT_CONFIG].iloc[0]

print("Numerical leader:", best["config"])
print(f"Tail NDCG@10: {best['tail_NDCG@10']:.5f}")
print(
    "Paired difference versus default: "
    f"{best['tail_delta_vs_default']:+.5f} "
    f"[{best['delta_CI_low']:+.5f}, {best['delta_CI_high']:+.5f}]"
)
print(
    "Cold NDCG@10: "
    f"{best['cold_NDCG@10']:.5f} "
    f"[{best['cold_NDCG_CI_low']:.5f}, {best['cold_NDCG_CI_high']:.5f}]"
)
print(
    "Default configuration — tail / cold NDCG@10: "
    f"{default_row['tail_NDCG@10']:.5f} / {default_row['cold_NDCG@10']:.5f}"
)

if best["config"] == DEFAULT_CONFIG:
    print("The original configuration is the numerical leader.")
elif best["delta_CI_low"] > 0:
    print("The leader's paired 95% interval is above zero: evidence of a validation gain over the default.")
else:
    print("The leader has the highest point estimate, but the paired interval does not establish a clear gain over the default.")

print("No model configuration has been frozen by this cell.")

Numerical leader: both_decade_1.0_idf_off
Tail NDCG@10: 0.00584
Paired difference versus default: +0.00055 [-0.00087, +0.00210]
Cold NDCG@10: 0.00565 [0.00236, 0.00943]
Default configuration — tail / cold NDCG@10: 0.00529 / 0.00563
The leader has the highest point estimate, but the paired interval does not establish a clear gain over the default.
No model configuration has been frozen by this cell.


In [20]:
reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

report_path = reports_dir / "content_ablation_validation.csv"
results_df.to_csv(report_path, index=False)
print("Saved preliminary validation results to:", report_path)

Saved preliminary validation results to: ..\reports\content_ablation_validation.csv


## Freeze content-v1

The selected point estimate uses equal genre and decade weights with IDF disabled. The paired interval versus the original IDF-weighted configuration crosses zero, so this is a validation choice—not a statistically established improvement.


In [21]:
X_content_v1 = make_feature_matrix(
    genre_weight=1.0,
    decade_weight=1.0,
    use_idf=False,
)
profiles_content_v1, n_liked_content_v1 = build_profiles_for(
    train, X_content_v1
)
content_v1_scorer = make_content_scorer(
    X_content_v1, profiles_content_v1
)
MODELS["content"] = content_v1_scorer

assert X_content_v1.shape == (n_items, 31)
assert profiles_content_v1.shape == (n_users, 31)
assert np.isfinite(X_content_v1).all()

feature_dir = Path("../data/interim/hybrid_features")
feature_dir.mkdir(parents=True, exist_ok=True)
X_content_v1_sparse = sp.csr_matrix(X_content_v1, dtype=np.float32)
sp.save_npz(feature_dir / "content_v1.npz", X_content_v1_sparse)

print("Frozen content-v1:", X_content_v1_sparse.shape)
print("Saved:", feature_dir / "content_v1.npz")


Frozen content-v1: (17241, 31)
Saved: ..\data\interim\hybrid_features\content_v1.npz


## Final validation report

Compare the frozen content-v1 features with random ranking and popularity in both the restricted rarity pool (View B) and the full catalogue (View A).


In [22]:
official_rows = []

N_BOOTSTRAP_OFFICIAL = 1000
bootstrap_rng = np.random.default_rng(SEED)
bootstrap_indices = {}

for view, restrict in [("B", True), ("A", False)]:
    for slice_name in ("cold", "tail"):

        evaluated, n_eval_users = eval_slice(
            slice_name,
            restrict=restrict,
            names=["random", "popularity", "content"],
        )

        # Use the same bootstrap samples for all models
        # and both views of the same rarity slice.
        if slice_name not in bootstrap_indices:
            bootstrap_indices[slice_name] = bootstrap_rng.integers(
                0,
                n_eval_users,
                size=(N_BOOTSTRAP_OFFICIAL, n_eval_users),
            )

        idx = bootstrap_indices[slice_name]

        for model_name, metrics in evaluated.items():
            model_label = (
                "content-v1" if model_name == "content"
                else model_name
            )

            for metric_name in ("NDCG@10", "Recall@10"):
                mean, low, high = ci(metrics[metric_name], idx)

                official_rows.append({
                    "model": model_label,
                    "comparison": "model_metric",
                    "view": view,
                    "slice": slice_name,
                    "n_users": n_eval_users,
                    "metric": metric_name,
                    "estimate": float(mean),
                    "ci95_low": float(low),
                    "ci95_high": float(high),
                })

        # Paired NDCG differences: content-v1 minus each baseline.
        for baseline_name in ("random", "popularity"):
            delta = (
                evaluated["content"]["NDCG@10"]
                - evaluated[baseline_name]["NDCG@10"]
            )

            mean, low, high = ci(delta, idx)

            official_rows.append({
                "model": "content-v1",
                "comparison": f"content-v1 minus {baseline_name}",
                "view": view,
                "slice": slice_name,
                "n_users": n_eval_users,
                "metric": "NDCG@10 difference",
                "estimate": float(mean),
                "ci95_low": float(low),
                "ci95_high": float(high),
            })

content_v1_val = pd.DataFrame(official_rows)

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

content_v1_val.to_csv(
    reports_dir / "content_v1_val.csv",
    index=False,
)

display(content_v1_val.round(6))
print("Official content-v1 validation report saved.")


,model,comparison,view,slice,n_users,metric,estimate,ci95_low,ci95_high
0,random,model_metric,B,cold,676,NDCG@10,0.000661,0.000257,0.001104
1,random,model_metric,B,cold,676,Recall@10,0.001405,0.000592,0.002293
2,popularity,model_metric,B,cold,676,NDCG@10,0.003596,0.000000,0.008671
3,popularity,model_metric,B,cold,676,Recall@10,0.004438,0.000000,0.010355
4,content-v1,model_metric,B,cold,676,NDCG@10,0.005647,0.002224,0.009882
5,content-v1,model_metric,B,cold,676,Recall@10,0.011144,0.004585,0.018984
6,content-v1,content-v1 minus random,B,cold,676,NDCG@10 difference,0.004986,0.001629,0.009293
7,content-v1,content-v1 minus popularity,B,cold,676,NDCG@10 difference,0.002051,-0.004280,0.007759
8,random,model_metric,B,tail,2144,NDCG@10,0.000514,0.000313,0.000747
9,random,model_metric,B,tail,2144,Recall@10,0.001000,0.000619,0.001436


Official content-v1 validation report saved.


Content-v1 is kept as the metadata representation for the LightFM experiment. It is not a standalone replacement for collaborative filtering; the next notebook tests whether LightFM can make use of these features.
